## Transformation
  - Rag 파이프라인에서 중요한 전처리 과정인 'Transformation'을 다룸. Transformation은 질의응답(Question-Answering) 및 생성(Generation)을 효과적으로 수행하기 위해 텍스트를 다양한 방식으로 나누는(Chunking)과정
  - Rag 모델은 대용량 텍스트 데이터에서 필요한 정보를 검색(Retrieval)한 뒤, 검색 결과를 입력으로 하여 응답(Generation)을 생성하는 구조. 이때, 문서(또는 여러 형태의 텍스트)를 어떻게 분할(Chunking)하고, 어떤 임베딩을 사용해 의미를 추출하느냐에 따라서 모델의 성능에 큰 영향을 준다.

In [ ]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

In [ ]:
#!pip install langchain==0.1.16
!pip install langchain

In [ ]:
!pip install faiss-gpu-cu12

In [1]:
import os
os.environ["API_KEY"] = ""

In [ ]:
from dotenv import load_dotenv
load_dotenv(override = True)

False

## CharacterTextSplitter 청킹
 - 텍스트를 일정 길이(캐릭터 수)단위로 분할하는 가장 단순한 접근 방법

In [ ]:
# pdf 파일 경로 설정
file_path = "paper1.pdf"

from langchain_community.document_loaders import PyPDFLoader

# pdf 로더 객체
loader = PyPDFLoader(file_path)

# pdf의 각 페이지를 저장할 리스트
pages = []

# 비동기 방식으로 pdf 페이지 로드
async for page in loader.alazy_load():
  pages.append(page)

In [ ]:
# characterTextSplitter 불러오기
from langchain_text_splitters import CharacterTextSplitter

# 텍스트 분할기
text_splitter = CharacterTextSplitter(
    separator="\n", # 문단 단위로 분할
    chunk_size=500,   # 하나의 청크(조각) 크기를 500자로 설정
    chunk_overlap=200, # 청크 간 200자 겹치게 설정
    length_function=len, # 텍스트 길이를 측정하는 함수 (len 사용)
    is_separator_regex = False, # seperator를 정규식이 아닌 단순 문자열로
)

# PDF에서 로드한 데이터를 텍스트 청크로 분할
texts = text_splitter.split_documents(pages)

print(f"{texts[0].metadata}")
print(texts[1].page_content) # 두 번째 청크의 내용
print("- "*50)
print(texts[3].page_content) # 세 번째 청크의 내용

{'source': 'paper1.pdf', 'page': 0}
이 연구는 한국노동패널(KLIPS) 자료를 활용하여 2017~2020년 동안 최저임금 인상이 정규직ㆍ비정규직 임금격차에 미친 영향을 실증적으로 분석하였다. 분석 방법으로는 이벤트 스터디(event study)와 삼중차분(Difference-in-Difference-in-Differences:DDD) 모형을 병행하였다. 이벤트 스터디 분석에서는 2017년을 기준연도로 설정하고, 상대연도별 시점 더미를 구성하여 정책 시행 이전의 평행추세(parallel trends) 가정을 검증한 후 정책 시행 이후의 동태적 효과를 추정하였다. 이어 DDD 분석에서는 최저임금 인상폭이 크게 확대된 시기(2017~2018, 2019년)를 대상으로 하여, 최저임금 미만ㆍ영향ㆍ차상위 임금집단 간의 차별적 변화를 비교하였다. 분석결과, 정책 시행 이전 기간에는 정규직과 비정규직의 임금추세가 유사하여 평행추세 가정이 충족되었으나, 정책 시행 이후 단기에는 유의한 변화가 없었고, 시행 2년 차(2020년)에 이르러 비정규직의 임금 상승폭이 정규직에 비해 유의하게 낮아지는 음(–)의 효과가 관찰되었다. 또한 DDD 분석에서도 정책노출 임금대(최저임금 미만ㆍ영향집단)에서 정규직–비정규직 간 임금격차의 추가적 축소 효과는 통계적으로 유의하지 않았다. 이러한 결과는 최저임금 인상이 단기적으로는 저임금층 임금 개선에 기여하였으나, 시간이 경과함에 따라 고용형태 간 임금격차 완화 효과가 약화되었음을 시사한다.핵심용어:최저임금 인상, 고용형태별 임금격차, 한국노동패널, 삼중차분, 이벤트 스터디논문접수일:2025년 8월 19일, 심사의뢰일:2025년 8월 19일, 심사완료일:2025년 10월 21일* 경북대학교 데이터사이언스학과, 석사과정(jinmin1569@nate.com)
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, # 하나의 청크 크기를 500자로 설정
    chunk_overlap=200, # 청크 간 200자 겹치게 설정(문맥 유지 목적)
    length_function=len, # 텍스트 길이를 측정하는 함수(len 사용)
    is_separator_regex=False,
)

# pdf에서 로드한 데이터를 텍스트 청크로 분할
texts = text_splitter.split_documents(pages)

print(f"{texts[1].metadata}") # 두 번째 청크의 메타데이터
print(texts[1].page_content)
print("-"*50)
print(f"{texts[4].metadata}")
print(texts[5].page_content)

{'source': 'paper1.pdf', 'page': 0}
이 연구는 한국노동패널(KLIPS) 자료를 활용하여 2017~2020년 동안 최저임금 인상이 정규직ㆍ비정규직 임금격차에 미친 영향을 실증적으로 분석하였다. 분석 방법으로는 이벤트 스터디(event study)와 삼중차분(Difference-in-Difference-in-Differences:DDD) 모형을 병행하였다. 이벤트 스터디 분석에서는 2017년을 기준연도로 설정하고, 상대연도별 시점 더미를 구성하여 정책 시행 이전의 평행추세(parallel trends) 가정을 검증한 후 정책 시행 이후의 동태적 효과를 추정하였다. 이어 DDD 분석에서는 최저임금 인상폭이 크게 확대된 시기(2017~2018, 2019년)를 대상으로 하여, 최저임금 미만ㆍ영향ㆍ차상위 임금집단 간의 차별적 변화를 비교하였다. 분석결과, 정책 시행 이전 기간에는 정규직과 비정규직의 임금추세가 유사하여 평행추세 가정이 충족되었으나, 정책 시행 이후 단기에는 유의한 변화가 없었고, 시행 2년 차(2020년)에
--------------------------------------------------
{'source': 'paper1.pdf', 'page': 0}
  노동정책연구ㆍ2025년 제25권 제4호96 I. 서 론본 연구는 최저임금의 급격한 인상이 고용형태별(정규직ㆍ비정규직) 임금격차에 어떠한 영향을 미쳤는지를 실증적으로 규명하는 데 목적을 둔다. 우리나라 최저임금 제도는 1988년 도입 이래 저임금 근로자를 보호하고 임금분포 하단의 불평등을 완화하는 중요한 정책수단으로 기능해왔으며, 특히 2018년에는 역대 최고 수준인 16.4%의 인상률이 적용되면서 정책 효과를 둘러싼 사회적 논쟁이 한층 고조되었다. 본 연구는 이러한 제도적 변화가 임금분포 하단과 고용형태별 임금구조에 남긴 영향을 계량적으로 분석하고자 한다.최저임금의 효과에 관한 국제 연구는 상반된 결과를 제시한다. Cengiz et al. (2019)은 미국의 대규

In [ ]:
from langchain_text_splitters import(
    Language,
    RecursiveCharacterTextSplitter,
)

# 지원되는 언어 목록 출력
print([e.value for e in Language])

['cpp', 'go', 'java', 'kotlin', 'js', 'ts', 'php', 'proto', 'python', 'rst', 'ruby', 'rust', 'scala', 'swift', 'markdown', 'latex', 'html', 'sol', 'csharp', 'cobol', 'c', 'lua', 'perl', 'haskell']


In [ ]:
# python 코드에 대한 기본적인 구분자 확인
RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)

['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ', '']

In [ ]:
# 샘플 Python 코드
PYTHON_CODE = """
class hello:
def hello_world():
  print("Hello, World!")

# Call the function
hello_world()
"""

# Python 언어에 최적화된 text_splitter 생성
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=50,
    chunk_overlap=0,
)
# Python 코드 문서를 분할하여 생성
python_docs = python_splitter.create_documents([PYTHON_CODE])
python_docs

[Document(page_content='class hello:'),
 Document(page_content='def hello_world():\n  print("Hello, World!")'),
 Document(page_content='# Call the function\nhello_world()')]

## Markdown
 - 헤더나 문서 구조를 기준으로 텍스트를 분할하여, 문서 구조를 활용하는 방법


In [ ]:
markdown_text = """
# 🦜🔗 LangChain

⚡ Building applications with LLMs through composability ⚡

## What is LangChain?

# Hopefully this code block isn't split
LangChain is a framework for...

As an open-source project in a rapidly developing field, we are extremely open to contributions.
"""
# Langchain의 RecursiveCharacterTextSplitter를 사용하여 Markdown 텍스트 분할
md_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=60,
    chunk_overlap=0
)

# Markdown 문서를 청크로 나누기
md_docs = md_splitter.create_documents([markdown_text])
md_docs

[Document(page_content='# 🦜🔗 LangChain'),
 Document(page_content='⚡ Building applications with LLMs through composability ⚡'),
 Document(page_content='## What is LangChain?'),
 Document(page_content="# Hopefully this code block isn't split"),
 Document(page_content='LangChain is a framework for...'),
 Document(page_content='As an open-source project in a rapidly developing field, we'),
 Document(page_content='are extremely open to contributions.')]

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Markdown 형식의 샘플 텍스트
markdown_document = "# Foo\n\n  ## Bar\n\nHi this is Jim\n\nHi this is Joe\n\n ### Boo \n\n Hi this is Lance \n\n ## Baz\n\n Hi this is Molly"

# 헤더를 기준으로 Markdown을 분할하기 위한 규칙 설정
headers_to_split_on = [
   ("#", "Header 1"),  # '#'은 Header 1로 분류
   ("##", "Header 2"), # '##'은 Header 2로 분류
   ("###","Header 3"), # '###'는 Header 3로 분류
]

# 헤더 기반으로 Markdown을 분할하는 Splitter 생성
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

# 설정한 헤더를 기준으로 Markdown을 분할
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

[Document(page_content='Hi this is Jim  \nHi this is Joe', metadata={'Header 1': 'Foo', 'Header 2': 'Bar'}),
 Document(page_content='Hi this is Lance', metadata={'Header 1': 'Foo', 'Header 2': 'Bar', 'Header 3': 'Boo'}),
 Document(page_content='Hi this is Molly', metadata={'Header 1': 'Foo', 'Header 2': 'Baz'})]

## 시맨틱 청킹
  - 텍스트의 의미를 기반으로 청크를 생성해, 문맥적으로 연관된 내용을 하나의 청크로 묶는 접근법

In [ ]:
!pip install --q langchain_experimental

In [ ]:
!pip install langchain-openai

In [ ]:
!pip install -U simsimd langchain-community

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

file_path = 'paper2.pdf'

from langchain_community.document_loaders import PyPDFLoader
import os
os.environ["OPENAI_API_KEY"] = ""
# pdf 로더 객체 생성
loader = PyPDFLoader(file_path)

# PDF의 각 페이지를 저장할 리스트
pages = []

# 비동기 방식으로 pdf의 각 페이지를 로드(async for 사용)
async for page in loader.alazy_load():
  pages.append(page)

#SemanticChunker를 사용하여 의미 기반으로 텍스트를 분할
text_splitter = SemanticChunker(OpenAIEmbeddings())

# 문서를 의미적 청킹(semantic chunking) 수행
docs = text_splitter.split_documents(pages)

print(docs[3].page_content)

연구방법3.1 연구방법본연구는국내대표기업리뷰플랫폼인잡플래닛(JobPlanet)에게시된제조·화학,IT,서비스업종의전·현직재직자리뷰데이터를기반으로,조직구성원들의직무및조직에대한인식유형을지도·비지도학습기반의군집화기법을통해분류하고자하였다.이를위해BERT기반으로임베딩된텍스


In [ ]:
print(docs[0].page_content)

잡플래닛데이터를활용한업종별긍정평가예측및변수중요도분석:군집기반머신러닝접근19
잡플래닛데이터를활용한업종별긍정평가예측및변수중요도분석:군집기반머신러닝접근진민준*
< 국문요약 >[연구목적]본연구는잡플래닛에게시된국내기업리뷰데이터를활용하여제조/화학,서비스,IT업종종사자들의조직인식성향을군집화하고,각군집내긍정평가여부를예측하는머신러닝기반분류모델을구축하며,이를통해산업별이질적구성원집단의존재를실증적으로규명하고긍정인식에영향을미치는주요요인을도출하는데목적이있다.[연구방법]분석은비지도학습기반의군집화와지도학습기반의이진분류로구성되며,WardLinkage와K-Means를비교적용한후GradientBoosting,NeuralNetwork,AdaBoost등다수의머신러닝모델을통해군집별긍정평가여부를예측하고,SHAP분석을통해변수별영향력을정량적으로해석하였다.[연구결과]분석결과,군집별로최적의분류모델이상이하게나타났으며,특히비계층적군집화(K-Means)를적용한경우대부분의산업군과군집에서더우수한예측성능을보였다.SHAP분석에서는‘기업추천여부’가모든업종에서가장중요한변수로확인되었고,IT업종에서는‘승진기회’에대한낮은평점이부정예측에크게기여하여경력개발기회에대한불만이조직평가에중요한영향을미치는것으로나타났다.서비스업에서는‘경영진’에대한인식이긍정평가에가장큰영향을미쳤으며,‘워라밸’은모든산업군에서상대적으로낮은예측기여도를보였다.[연구의시사점]본연구는비정형리뷰텍스트와정량데이터를통합분석하고,설명가능한인공지능기반의예측모형을적용함으로써군집별맞춤형HR진단과전략수립의실증적기반을제공하였다.이는기업이이탈위험군을조기식별하고,구성원특성에기반한맞춤형인사전략을설계하는데유용한분석프레임워크로활용될수있으며,HRM분야에서이론적·실무적으로기여할수있을것으로기대된다.핵심주제어:잡플래닛,Wardlinkage,K-Means,긍정평가예측,SHAP,업종별분석
논문접수일:2025년5월11일수정일:2025년6월21일게재확정일:2025년7월26일*석사과정,경북대학교,데이터사이언스학과,jinmin1569@nate.com
대한경영정보학회「경영과정보연구」제44권제3호2025년9월http:/

In [ ]:
print(f"총 {len(docs)}개 만큼의 문서로 청킹되었습니다.")
print([len(i.page_content) for i in docs])

총 26개 만큼의 문서로 청킹되었습니다.
[1040, 1397, 1017, 146, 723, 1351, 629, 1176, 949, 830, 1309, 1407, 1341, 1252, 1256, 1326, 1686, 1568, 2400, 888, 800, 295, 592, 1500, 2411, 2312]


In [ ]:
# 각 청크의 메타데이터 및 내용 출력
for i in docs:
  print(i.metadata)
  print(i.page_content)
  print("-"*50)

{'producer': 'Hancom PDF 1.3.0.547', 'creator': 'Hwp 2024 13.0.0.564', 'creationdate': '2025-10-02T13:14:21+09:00', 'author': '용강중', 'moddate': '2025-10-02T13:14:21+09:00', 'pdfversion': '1.4', 'source': 'paper2.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}
잡플래닛데이터를활용한업종별긍정평가예측및변수중요도분석:군집기반머신러닝접근19
잡플래닛데이터를활용한업종별긍정평가예측및변수중요도분석:군집기반머신러닝접근진민준*
< 국문요약 >[연구목적]본연구는잡플래닛에게시된국내기업리뷰데이터를활용하여제조/화학,서비스,IT업종종사자들의조직인식성향을군집화하고,각군집내긍정평가여부를예측하는머신러닝기반분류모델을구축하며,이를통해산업별이질적구성원집단의존재를실증적으로규명하고긍정인식에영향을미치는주요요인을도출하는데목적이있다.[연구방법]분석은비지도학습기반의군집화와지도학습기반의이진분류로구성되며,WardLinkage와K-Means를비교적용한후GradientBoosting,NeuralNetwork,AdaBoost등다수의머신러닝모델을통해군집별긍정평가여부를예측하고,SHAP분석을통해변수별영향력을정량적으로해석하였다.[연구결과]분석결과,군집별로최적의분류모델이상이하게나타났으며,특히비계층적군집화(K-Means)를적용한경우대부분의산업군과군집에서더우수한예측성능을보였다.SHAP분석에서는‘기업추천여부’가모든업종에서가장중요한변수로확인되었고,IT업종에서는‘승진기회’에대한낮은평점이부정예측에크게기여하여경력개발기회에대한불만이조직평가에중요한영향을미치는것으로나타났다.서비스업에서는‘경영진’에대한인식이긍정평가에가장큰영향을미쳤으며,‘워라밸’은모든산업군에서상대적으로낮은예측기여도를보였다.[연구의시사점]본연구는비정형리뷰텍스트와정량데이터를통합분석하고,설명가능한인공지능기반의예측모형을적용함으로써군집별맞춤형HR진단과전략수립의실증적기반을제공